# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print basic info
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published date: {dataset.metadata.datePublished}")
print(f"Cite as: {dataset.metadata.citeAs}")
print('---')
print('Keywords:', dataset.metadata.keywords)

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant metadata to list the record sets and their associated field (column) `@id`s. All references use their `@id`.

In [ ]:
from mlcroissant.types import RecordSet

# List all RecordSets with their @id and columns
def print_recordsets(ds):
    print('Available RecordSets:')
    for rs in ds.metadata.record_sets:
        print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
        print('  Columns:')
        for col in rs.columns:
            print(f"     - {col.name}, @id: {col.id}, dataType: {col.data_type}")
        print()
    print('---')
    if len(ds.metadata.record_sets) == 0:
        print('No explicit record sets listed in .metadata.record_sets; will try to enumerate from .record_sets property (for backward compatibility).')
        if hasattr(ds, 'record_sets'):
            for rs in ds.record_sets:
                print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
                print('  Columns:')
                for col in rs.columns:
                    print(f"     - {col.name}, @id: {col.id}, dataType: {col.data_type}")
                print()
        else:
            print('No record sets available.')
        print('---')

# Explore record sets in the dataset
print_recordsets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Note: In Croissant, record sets may have complex IDs. Be sure to use the full `@id`. We'll extract all available record sets and demonstrate converting each to pandas DataFrame.

In [ ]:
# Find all record set IDs
record_sets = [rs.id for rs in dataset.metadata.record_sets]
if not record_sets:
    # Try loading from older property
    record_sets = [rs.id for rs in getattr(dataset, 'record_sets', [])]

print('Record set @ids:', record_sets)

dataframes = {}

for record_set in record_sets:
    records_iter = dataset.records(record_set=record_set)
    records = list(records_iter)
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set '{record_set}'")

# Preview the columns for a chosen record set
chosen_record_set = record_sets[0]
print(f"Columns in {chosen_record_set}:")
print(dataframes[chosen_record_set].columns.tolist())
dataframes[chosen_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis. All column references use their `@id`.

In [ ]:
# Choose a numeric field for analysis:
# Find the first numeric column in the chosen record set
numeric_field = None
group_field = None
for rs in dataset.metadata.record_sets:
    if rs.id == chosen_record_set:
        for col in rs.columns:
            if col.data_type in ['Number', 'Float', 'Integer']:
                numeric_field = col.id
                break
        
        # Try to find a non-numeric column for grouping
        for col in rs.columns:
            if col.data_type in ['Text', 'String']:
                group_field = col.id
                break
        break

print('Numeric field @id:', numeric_field)
print('Group field @id:', group_field)

df = dataframes[chosen_record_set]

# Handle missing values and convert to numeric if necessary
if numeric_field and numeric_field in df.columns:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].notna().sum() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    mean_val = filtered_df[numeric_field].mean()
    std_val = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean_val) / std_val
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field if available (e.g., description field)
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {chosen_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field and group_field in df.columns:
        # Show boxplot by group (if groups are not overly sparse)
        plt.figure(figsize=(12,6))
        group_counts = df[group_field].value_counts()
        top_groups = group_counts.nlargest(12).index  # Show only top N groups
        subset = df[df[group_field].isin(top_groups)]
        sns.boxplot(data=subset, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (top groups)")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the Croissant FAIR^2 dataset describing mass spectrometry profiles of extracellular vesicles from human mast cells. We:
- Loaded dataset metadata, record sets, and columns using their Croissant `@id` references
- Extracted data into pandas DataFrames for flexible analysis
- Applied exploratory data transformations such as filtering and normalization
- Visualized numeric features and compared means across sample groups.

**You can further extend this notebook by applying advanced analyses and integrating with other tools, leveraging the power of `mlcroissant` and the interoperable Croissant schema.**